In [3]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = "."
PRED_PATH = "/storage/gwl-interpolation/outputs/TFT/TFT_in52_out16_ep50_bs4096_stat1_seed40_full_merged/predictions/pred.parquet"
OBS_PATH  = "/storage/data/sample.csv"
CUTOFF    = pd.Timestamp("2024-01-01")

pred = pq.read_table(PRED_PATH).to_pandas()
pred = pred.rename(columns={"gws":"gws_forecast"})
gws_bb = pd.read_csv(OBS_PATH)
gws_bb["datum"] = pd.to_datetime(gws_bb["datum"])

lookup_ids = pd.DataFrame(gws_bb["id"].unique(), columns=["id"]).reset_index()

df = (pred.merge(lookup_ids, on="index")
          .merge(gws_bb[["id","datum","gws"]], on=["id","datum"], how="left")
          .rename(columns={"gws":"gws_true"}))

df["horizon"] = ((df["datum"] - df["startzeitpunkt"]) / pd.Timedelta(weeks=1)).astype(int) + 1

df["gws_forecast"] = df["gws_forecast"].astype(np.float32).round(2)
df["difference"] = (df["gws_forecast"] - df["gws_true"]).abs().round(2)

hist = gws_bb[gws_bb["datum"] < CUTOFF].copy()
hist["woy"] = hist["datum"].dt.isocalendar().week.astype(int)
clim = hist.groupby(["id","woy"], as_index=False)["gws"].mean().rename(columns={"gws":"clim_base"})

df["woy"] = df["datum"].dt.isocalendar().week.astype(int)
df = df.merge(clim, on=["id","woy"], how="left")

well_mean = hist.groupby("id")["gws"].mean().rename("well_mean")
df = df.merge(well_mean, on="id", how="left")
df["clim_base"] = df["clim_base"].fillna(df["well_mean"])
df.drop(columns=["well_mean"], inplace=True)

/tmp/ipykernel_1610/430960588.py:13: DtypeWarning: Columns (54,55) have mixed types. Specify dtype option on import or set low_memory=False.
  gws_bb = pd.read_csv(OBS_PATH)


In [4]:
def mae(y, yhat): return np.mean(np.abs(yhat - y))
def rmse(y, yhat): return np.sqrt(np.mean((yhat - y)**2))
def nse(y, yhat):
    denom = np.sum((y - np.mean(y))**2)
    return np.nan if denom == 0 else 1 - np.sum((yhat - y)**2) / denom
def kge(y, yhat):
    r = np.corrcoef(y, yhat)[0,1]
    alpha = np.std(yhat, ddof=1) / np.std(y, ddof=1)
    beta  = np.mean(yhat) / np.mean(y)
    return 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)
def rmbe(y, yhat):
    mbe = np.mean(yhat - y)
    return mbe / np.mean(y)
def nrmse(y, yhat, how="range"):
    r = rmse(y, yhat)
    if how == "range":   denom = np.max(y) - np.min(y)
    elif how == "mean":  denom = np.mean(y)
    elif how == "std":   denom = np.std(y, ddof=1)
    else: raise ValueError("how must be 'range','mean','std'")
    return np.nan if denom == 0 else r / denom


mask = np.isfinite(df["gws_true"]) & np.isfinite(df["gws_forecast"])
y, yhat = df.loc[mask, "gws_true"].to_numpy(), df.loc[mask, "gws_forecast"].to_numpy()
metrics = {
    "KGE":  kge(y, yhat),
    "MAE":  mae(y, yhat),
    "NSE":  nse(y, yhat),
    "RMSE": rmse(y, yhat),
    "NRMSE_range": nrmse(y, yhat, "range"),
    "RMBE": rmbe(y, yhat),
}
pd.Series(metrics)

KGE            0.999523
MAE            0.073925
NSE            0.999962
RMSE           0.118174
NRMSE_range    0.002560
RMBE          -0.000272
dtype: float64

In [5]:
"""
Results from first run with leakage from future covariates:

KGE            0.999694
MAE            0.055127
NSE            0.999983
RMSE           0.080789
NRMSE_range    0.001750
RMBE           0.000194

"""

'\nResults from first run with leakage from future covariates:\n\nKGE            0.999694\nMAE            0.055127\nNSE            0.999983\nRMSE           0.080789\nNRMSE_range    0.001750\nRMBE           0.000194\n\n'

In [6]:
def skill(group):
    rmse_model = rmse(group["gws_true"], group["gws_forecast"])
    rmse_clim = rmse(group["gws_true"], group["clim_base"])
    return np.nan if rmse_clim == 0 else 1 - rmse_model / rmse_clim

test = df[df["datum"] >= CUTOFF].copy()
by_hor = test.groupby("horizon", dropna=False).apply(skill)
by_well = test.groupby("id", dropna=False).apply(skill)

print("RMSE by horizon:")
pd.Series(by_hor.to_dict())

RMSE by horizon:


/tmp/ipykernel_1610/3031827182.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_hor = test.groupby("horizon", dropna=False).apply(skill)
/tmp/ipykernel_1610/3031827182.py:8: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  by_well = test.groupby("id", dropna=False).apply(skill)


1     0.854868
2     0.811523
3     0.770403
4     0.726286
5     0.688057
6     0.675285
7     0.638740
8     0.602443
9     0.579996
10    0.558499
11    0.547780
12    0.546046
13    0.550347
14    0.554194
15    0.559903
16    0.559452
dtype: float64

In [7]:
print("Median of RMSE by well:") 
float(by_well.median())

Median of RMSE by well:


0.7079782810744436

In [8]:
df.head(10)

,datum,gws_forecast,index,startzeitpunkt,id,gws_true,horizon,difference,woy,clim_base
0,2021-01-04,78.559998,0,2021-01-04,LFU_25470023,78.58,1,0.02,1,79.212174
1,2021-01-11,78.580002,0,2021-01-04,LFU_25470023,78.56,2,0.02,2,79.222609
2,2021-01-18,78.599998,0,2021-01-04,LFU_25470023,78.57,3,0.03,3,79.233043
3,2021-01-25,78.610001,0,2021-01-04,LFU_25470023,78.58,4,0.03,4,79.234348
4,2021-02-01,78.610001,0,2021-01-04,LFU_25470023,78.57,5,0.04,5,79.241304
5,2021-02-08,78.629997,0,2021-01-04,LFU_25470023,78.53,6,0.10,6,79.244783
6,2021-02-15,78.639999,0,2021-01-04,LFU_25470023,78.58,7,0.06,7,79.243478
7,2021-02-22,78.650002,0,2021-01-04,LFU_25470023,78.63,8,0.02,8,79.260870
8,2021-03-01,78.650002,0,2021-01-04,LFU_25470023,78.61,9,0.04,9,79.278696
9,2021-03-08,78.660004,0,2021-01-04,LFU_25470023,78.66,10,0.00,10,79.315652


In [9]:
import numpy as np
from math import sqrt

persist = (gws_bb[["id","datum","gws"]]
           .rename(columns={"datum":"startzeitpunkt","gws":"persist_base"}))

df_p = df.merge(persist, on=["id","startzeitpunkt"], how="left")

def rmse(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() == 0: return np.nan
    return sqrt(np.mean((a[m]-b[m])**2))

# overall skill vs persistence
rmse_model   = rmse(df_p["gws_forecast"], df_p["gws_true"])
rmse_persist = rmse(df_p["persist_base"], df_p["gws_true"])
skill_overall = np.nan if not np.isfinite(rmse_persist) or rmse_persist==0 else 1 - rmse_model/rmse_persist
print({"RMSE_model": rmse_model, "RMSE_persist": rmse_persist, "skill_vs_persist": skill_overall})

# by horizon
by_h = (df_p.groupby("horizon", dropna=False)
          .apply(lambda g: (np.nan if rmse(g["persist_base"], g["gws_true"]) in [0, np.nan]
                            else 1 - rmse(g["gws_forecast"], g["gws_true"])
                                     / rmse(g["persist_base"], g["gws_true"]))))

print("skill by horizon:")
print(by_h)

# by well
by_well = (df_p.groupby("id", dropna=False)
             .apply(lambda g: (np.nan if rmse(g["persist_base"], g["gws_true"]) in [0, np.nan]
                               else 1 - rmse(g["gws_forecast"], g["gws_true"])
                                        / rmse(g["persist_base"], g["gws_true"]))))

print("median skill by well:", np.nanmedian(by_well.values))


{'RMSE_model': 0.1181738344451362, 'RMSE_persist': 0.12082308706907977, 'skill_vs_persist': 0.02192670861346946}
skill by horizon:
horizon
1          NaN
2    -0.397803
3    -0.177094
4    -0.112930
5    -0.049931
6    -0.006816
7    -0.008896
8    -0.002104
9     0.014236
10    0.023413
11    0.036205
12    0.050624
13    0.064072
14    0.068568
15    0.072719
16    0.071352
dtype: float64
median skill by well: 0.11187336727214237


/tmp/ipykernel_1610/1576013974.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (np.nan if rmse(g["persist_base"], g["gws_true"]) in [0, np.nan]
/tmp/ipykernel_1610/1576013974.py:32: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: (np.nan if rmse(g["persist_base"], g["gws_true"]) in [0, np.nan]
